# Neural Pattern Associator (NPA)

Paper: [Within-basket Recommendation via Neural Pattern Associator](https://arxiv.org/abs/2401.16433)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import os

# os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
# os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# torch.set_num_threads(1)
# os.environ['OMP_NUM_THREADS'] = '1'
# os.environ['MKL_NUM_THREADS'] = '1'

print(f"Torch threads: {torch.get_num_threads()}")

Using device: cuda
Torch threads: 14


In [3]:
cd ..

/home/jupyter/work/resources/hse-masters-thesis-2026


In [4]:
import sys
import os

# project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
# if project_root not in sys.path:
#     sys.path.insert(0, project_root)
# os.chdir(project_root)

from tecd_retail_recsys.data import DataPreprocessor
from tecd_retail_recsys.models.npa import NPATrainer

In [5]:
dp = DataPreprocessor(
    day_begin=1082, 
    day_end=1308, 
    val_days=20, 
    test_days=20, 
    min_user_interactions=1, 
    min_item_interactions=20
)
train_df, val_df, test_df = dp.preprocess()

Starting data preprocessing...
Loading events from t_ecd_small_partial/dataset/small/retail/events
Loaded 267,043,972 total events
Loading items data from t_ecd_small_partial/dataset/small/retail/items.pq
Loaded 250,171 items with features: ['item_id', 'item_brand_id', 'item_category', 'item_subcategory', 'item_price', 'item_embedding']
Merged item features. Data shape: (267043972, 12)


IOStream.flush timed out


Filtered to 3,852,145 events with action_type='added_to_cart'
After filtering (min_user_interactions=1, min_item_interactions=20): 3,327,057 events, 84,830 users, 32,095 items
Created mappings: 84830 users, 32095 items
Temporal split - Train: days < 1269 (922,368 events), Val: days 1269-1288 (232,900 events), Test: days >= 1289 (227,959 events)
Users in each part (train, val, test) - 7422


In [6]:
import pandas as pd
import numpy as np

def group_by_baskets(df, time_window_hours=2, min_basket_size=2):
    """
    Group interactions into baskets by user using time windows.
    """
    
    if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df = df.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
    
    df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)
    
    user_baskets = {}
    
    for user_id, group in df.groupby('user_id'):
        group = group.sort_values('timestamp')
        timestamps = group['timestamp'].values
        items = group['item_id'].values
        time_delta = pd.to_timedelta(time_window_hours, unit='h')
        
        baskets = []
        current_basket = [items[0]]
        
        for i in range(1, len(items)):
            gap = pd.Timestamp(timestamps[i]) - pd.Timestamp(timestamps[i - 1])
            if gap <= time_delta:
                current_basket.append(items[i])
            else:
                if len(current_basket) >= min_basket_size:
                    seen = set()
                    deduped = []
                    for item in current_basket:
                        if item not in seen:
                            seen.add(item)
                            deduped.append(item)
                    baskets.append(deduped)
                current_basket = [items[i]]

        if len(current_basket) >= min_basket_size:
            seen = set()
            deduped = []
            for item in current_basket:
                if item not in seen:
                    seen.add(item)
                    deduped.append(item)
            baskets.append(deduped)
        
        if baskets:
            user_baskets[user_id] = baskets
    
    user_ids = list(user_baskets.keys())
    return user_baskets, user_ids


train_user_baskets, train_user_ids = group_by_baskets(train_df, time_window_hours=2, min_basket_size=2)
val_user_baskets, val_user_ids = group_by_baskets(val_df, time_window_hours=2, min_basket_size=2)
test_user_baskets, test_user_ids = group_by_baskets(test_df, time_window_hours=2, min_basket_size=2)

basket_sizes = [len(b) for baskets in train_user_baskets.values() for b in baskets]
seq_lengths = [len(baskets) for baskets in train_user_baskets.values()]

print(f"Users:                 {len(train_user_baskets)}")
print(f"Total baskets:         {len(basket_sizes)}")
print(f"Basket size:           median={np.median(basket_sizes):.0f}, "
      f"mean={np.mean(basket_sizes):.1f}, "
      f"max={np.max(basket_sizes)}")
print(f"Baskets per user:      median={np.median(seq_lengths):.0f}, "
      f"mean={np.mean(seq_lengths):.1f}, "
      f"max={np.max(seq_lengths)}")
print(f"Users with ≥2 baskets: {sum(1 for s in seq_lengths if s >= 2)} "
      f"({sum(1 for s in seq_lengths if s >= 2)/len(seq_lengths)*100:.0f}%)")


Users:                 7138
Total baskets:         120115
Basket size:           median=4, mean=6.6, max=124
Baskets per user:      median=11, mean=16.8, max=263
Users with ≥2 baskets: 6697 (94%)


In [ ]:
trainer = NPATrainer(
    train_user_baskets,

    embed_dim=512,           # размерность эмбеддингов
    n_layers=2,              # кол-во слоёв VQA (stacked)
    n_channels=2,            # кол-во параллельных VQA каналов в слое
    codebook_size=256,        # размер кодбука паттернов
    dropout=0.2,
    variant="MC",            # "SC" (squashed-context) или "MC" (multi-context)

    lr=1e-3,
    weight_decay=1e-5,
    batch_size=512,
    epochs=20,

    holdout_ratio=0.15,       # доля корзин для eval
    target_ratio=0.2,        # последние 20% корзины → ground truth
    eval_k=100,              # NDCG@100

    max_basket_size=30,
    seed=42,
    device="auto",           # "cpu", "cuda", "auto"
    verbose=True,
)

Items:         31823
Total baskets: 120115
Train baskets: 102098
Eval baskets:  18017
Device:        cuda
Model params:  44,427,344


In [24]:
trainer.train()

Epoch   1/20 │ loss=9.7158 │ NDCG@100=0.0359 │ Recall@100=0.1160
Epoch   2/20 │ loss=9.3846 │ NDCG@100=0.0463 │ Recall@100=0.1443
Epoch   3/20 │ loss=9.2001 │ NDCG@100=0.0525 │ Recall@100=0.1570
Epoch   4/20 │ loss=9.0557 │ NDCG@100=0.0555 │ Recall@100=0.1634
Epoch   5/20 │ loss=8.9215 │ NDCG@100=0.0589 │ Recall@100=0.1744
Epoch   6/20 │ loss=8.7807 │ NDCG@100=0.0616 │ Recall@100=0.1804
Epoch   7/20 │ loss=8.6544 │ NDCG@100=0.0640 │ Recall@100=0.1824
Epoch   8/20 │ loss=8.5163 │ NDCG@100=0.0639 │ Recall@100=0.1830
Epoch   9/20 │ loss=8.3612 │ NDCG@100=0.0631 │ Recall@100=0.1799
Epoch  10/20 │ loss=8.2109 │ NDCG@100=0.0648 │ Recall@100=0.1840
Epoch  11/20 │ loss=8.0258 │ NDCG@100=0.0656 │ Recall@100=0.1839
Epoch  12/20 │ loss=7.8460 │ NDCG@100=0.0644 │ Recall@100=0.1813
Epoch  13/20 │ loss=7.6840 │ NDCG@100=0.0643 │ Recall@100=0.1812
Epoch  14/20 │ loss=7.5286 │ NDCG@100=0.0652 │ Recall@100=0.1799
Epoch  15/20 │ loss=7.4037 │ NDCG@100=0.0652 │ Recall@100=0.1807
Epoch  16/20 │ loss=7.308

In [60]:
metrics = trainer.evaluate()

Eval  NDCG@100  = 0.0655
Eval  Recall@100 = 0.1822


In [27]:
model_path = "./models/npa/best_model.pt"

checkpoint = {
    "model_state_dict": trainer.model.state_dict(),
    "optimizer_state_dict": trainer.optimizer.state_dict(),
    "scheduler_state_dict": trainer.scheduler.state_dict(),
    "model_config": {
        "num_items": trainer.model.num_items,
        "embed_dim": trainer.model.embed_dim,
        "n_layers": len(trainer.model.layers),
        "n_channels": len(trainer.model.layers[0].channels),
        "codebook_size": trainer.model.layers[0].channels[0].codebook_size,
        "variant": trainer.model.variant,
    },
    "trainer_config": {
        "eval_k": trainer.eval_k,
        "target_ratio": trainer.target_ratio,
        "batch_size": trainer.batch_size,
    },
}
torch.save(checkpoint, model_path)

In [28]:
# инференс для конкретной корзины (аля realtime)
basket = [8591, 4010, 750]
recommendations = trainer.predict(basket, top_k=10)
print("Top-10 recommendations:")
for item_id, score in recommendations:
    print(f"  item {item_id:>6d}  score={score:.4f}")

Top-10 recommendations:
  item  11929  score=0.0147
  item   7833  score=0.0076
  item  24562  score=0.0057
  item   8306  score=0.0053
  item  27594  score=0.0052
  item  10943  score=0.0051
  item  18598  score=0.0050
  item  26491  score=0.0048
  item  11331  score=0.0045
  item  24321  score=0.0044


In [29]:
# разные стратегии рекомендаций на подвыборке пользователей

sample_val_user_baskets = {k:v for k,v in val_user_baskets.items() if k in list(val_user_baskets.keys())[:100]}

# Greedy
res_greedy = trainer.evaluate_autoregressive(
    sample_val_user_baskets,
    target_ratio=0.3,
    top_k=100,
    strategy="greedy",
)

# Nucleus sampling
res_nucleus = trainer.evaluate_autoregressive(
    sample_val_user_baskets,
    target_ratio=0.3,
    top_k=100,
    strategy="nucleus",
    top_p=0.9,
    temperature=0.7,
)
# Multirun top-k sampling
res_multi = trainer.evaluate_autoregressive(
    sample_val_user_baskets,
    target_ratio=0.3,
    top_k=100,
    strategy="top_k",
    top_k_sample=200,
    temperature=0.7,
    n_runs=10,
)

# Default (not autoregressive) sampling
res_single = trainer.evaluate_external(sample_val_user_baskets, target_ratio=0.3, top_k=100)

print("\n===== Сравнение =====")
print(f"Default (Not-AR):  NDCG={res_single['ndcg']:.4f}  Recall={res_single['recall']:.4f}")
print(f"AR greedy:         NDCG={res_greedy['ndcg']:.4f}  Recall={res_greedy['recall']:.4f}")
print(f"AR nucleus:        NDCG={res_nucleus['ndcg']:.4f}  Recall={res_nucleus['recall']:.4f}")
print(f"AR multi-run (10): NDCG={res_multi['ndcg']:.4f}  Recall={res_multi['recall']:.4f}")


Strategy:         greedy
Baskets evaluated: 309
NDCG@100:       0.0834
Recall@100:     0.1787
HitRate@100:    0.3333
Strategy:         nucleus
Baskets evaluated: 309
NDCG@100:       0.0382
Recall@100:     0.1097
HitRate@100:    0.2201
Strategy:         top_k (x10 runs)
Baskets evaluated: 309
NDCG@100:       0.0722
Recall@100:     0.1724
HitRate@100:    0.3269
Baskets evaluated: 309
NDCG@100:       0.0879
Recall@100:     0.1930

===== Сравнение =====
Default (Not-AR):  NDCG=0.0879  Recall=0.1930
AR greedy:         NDCG=0.0834  Recall=0.1787
AR nucleus:        NDCG=0.0382  Recall=0.1097
AR multi-run (10): NDCG=0.0722  Recall=0.1724


In [30]:
# evaluation with best strategy
best_strategy_metrics = trainer.evaluate_external(
    val_user_baskets,
    target_ratio=0.3,
    top_k=100
)

Baskets evaluated: 21141
NDCG@100:       0.0597
Recall@100:     0.1566


`Удалось добиться ndcg@100 = 0.0597 на задаче within-basket recommendation. Результат ниже, чем у персонализированных подходов, использующих историю покупок. Модель NPA не требует истории пользователя и опирается исключительно на паттерны совместных покупок, выученные из обучающих корзин. Это делает ее применимой в сценарии холодного старта, где персонализированные методы неработоспособны. Обученная модель работает в real-time режиме, адаптируя рекомендации по мере добавления товаров в корзину. Ограничение: модель не учитывает индивидуальные предпочтения пользователя и не способна рекомендовать товары, отсутствующие в обучающей выборке.`

In [64]:
from tecd_retail_recsys.utils import get_avg_recs_price, get_item_to_price
item_to_price = get_item_to_price(dp)

In [ ]:
# инференс на тестовой выборке

from tecd_retail_recsys.models.npa import ndcg_at_k, recall_at_k
from tqdm import tqdm

metrics = {
    'recall@10': [],
    'recall@100': [],
    'ndcg@10': [],
    'ndcg@100': [],
    'avg_price': []
}

trainer.model.eval()
for user_id in tqdm(list(test_user_baskets.keys())):
    
    user_baskets = test_user_baskets[user_id]
    for basket in user_baskets:
        if len(basket) < 3:
            continue
        basket_test_len = max(1, int(len(basket) * 0.2))
        train_part, val_part = basket[:-basket_test_len], basket[-basket_test_len:]
        if len(train_part) == 0 or len(val_part) == 0:
            continue
        val_preds = trainer.predict(train_part, top_k=100)
        val_recs = [rec for rec, score in val_preds]
        for item in val_recs:
            try:
                metrics['avg_price'].append(item_to_price[item])
            except KeyError:
                pass

        mapped_input = []
        seen = set()
        for item in train_part:
            if item in trainer.item2idx and item not in seen:
                seen.add(item)
                mapped_input.append(trainer.item2idx[item])

        mapped_target = []
        seen_t = set()
        for item in val_part:
            if item in trainer.item2idx and item not in seen_t:
                seen_t.add(item)
                mapped_target.append(trainer.item2idx[item])

        if len(mapped_input) == 0 or len(mapped_target) == 0:
            continue

        input_ids = torch.LongTensor([mapped_input]).to(trainer.device)
        mask = torch.ones(1, len(mapped_input), dtype=torch.bool, device=trainer.device)

        logits = trainer.model(input_ids, mask)

        logits[:, 0] = float("-inf")
        for idx in mapped_input:
            logits[0, idx] = float("-inf")

        targets = torch.LongTensor([mapped_target]).to(trainer.device)

        ndcg10 = ndcg_at_k(logits.clone(), targets, k=10)
        ndcg100 = ndcg_at_k(logits.clone(), targets, k=100)
        recall10 = recall_at_k(logits.clone(), targets, k=10)
        recall100 = recall_at_k(logits.clone(), targets, k=100)

        metrics['ndcg@10'].append(ndcg10)
        metrics['ndcg@100'].append(ndcg100)
        metrics['recall@10'].append(recall10)
        metrics['recall@100'].append(recall100)

100%|██████████| 6750/6750 [02:31<00:00, 44.53it/s]


In [83]:
# тестовые метрики

print(f"NDCG@10 = {np.mean(metrics['ndcg@10']):.3f}")
print(f"NDCG@100 = {np.mean(metrics['ndcg@100']):.3f}")
print(f"Recall@10 = {np.mean(metrics['recall@10']):.3f}")
print(f"Recall@100 = {np.mean(metrics['recall@100']):.3f}")
print(f"Avg rec price = {np.mean(metrics['avg_price']):.3f}")

NDCG@10 = 0.029
NDCG@100 = 0.051
Recall@10 = 0.046
Recall@100 = 0.145
Avg rec price = -3.965
